In [22]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid


In [23]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (129477, 46)
Val:   (43755, 46)
Test:  (12853, 46)


In [24]:
TARGET = "log_price"
DROP_COLS = ["ClosePrice", "CloseDate", "price_ratio", TARGET]
feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_val, y_val = val_df[feature_cols], val_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

RANDOM_STATE = 20260805

In [25]:
def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


## Outlier

In [26]:
lower_cap = train_df["price_ratio"].quantile(0.005)
upper_cap = train_df["price_ratio"].quantile(0.9995)
print(f"Lower cap: {lower_cap:.4f}, Upper cap: {upper_cap:.4f}")

train_mask = (train_df["price_ratio"] >= lower_cap) & (
    train_df["price_ratio"] <= upper_cap
)
val_mask = (val_df["price_ratio"] >= lower_cap) & (val_df["price_ratio"] <= upper_cap)
test_mask = (test_df["price_ratio"] >= lower_cap) & (
    test_df["price_ratio"] <= upper_cap
)

print(
    f"Train: {train_mask.sum()} / {len(train_df)} kept ({(1 - train_mask.mean()) * 100:.4f}% removed)"
)
print(
    f"Val: {val_mask.sum()} / {len(val_df)} kept ({(1 - val_mask.mean()) * 100:.4f}% removed)"
)
print(
    f"Test: {test_mask.sum()} / {len(test_df)} kept ({(1 - test_mask.mean()) * 100:.4f}% removed)"
)


Lower cap: 0.7893, Upper cap: 1.9807
Train: 128764 / 129477 kept (0.5507% removed)
Val: 43507 / 43755 kept (0.5668% removed)
Test: 12794 / 12853 kept (0.4590% removed)


In [27]:
X_train_no_outliers = X_train[train_mask]
y_train_no_outliers = y_train[train_mask]

X_val_no_outliers = X_val[val_mask]
y_val_no_outliers = y_val[val_mask]

X_test_no_outliers = X_test[test_mask]
y_test_no_outliers = y_test[test_mask]


## XGBoost

## Baseline with and without outliers

In [28]:
xgb_base = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base.fit(X_train, y_train)
evaluate(y_val, xgb_base.predict(X_val), "XGB base")


  XGB base | RMSE: $8,237,289  MAE: $284,916  R2: 0.0236  MAPE: 16.11%


{'rmse': 8237288.7267421335,
 'mae': 284916.1648652443,
 'r2': 0.023611924365872672,
 'mape': np.float64(16.105077621808135)}

In [29]:
xgb_base_no_outliers = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    xgb_base_no_outliers.predict(X_val_no_outliers),
    "XGB base without outliers",
)


XGB base without outliers | RMSE: $578,809  MAE: $181,277  R2: 0.8373  MAPE: 11.95%


{'rmse': 578808.6798516219,
 'mae': 181277.08293508805,
 'r2': 0.8372785559099081,
 'mape': np.float64(11.953563885632816)}

### hyperparameter tuning

In [30]:
param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [300, 600],
}


def grid_search_xgb(param_grid, X_train, y_train, X_val, y_val):
    results = []
    for params in ParameterGrid(param_grid):
        model = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rmse = root_mean_squared_error(np.exp(y_val), np.exp(pred))
        results.append({**params, "val_rmse": rmse})
    return pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)


xgb_results = grid_search_xgb(
    param_grid,
    X_train_no_outliers,
    y_train_no_outliers,
    X_val_no_outliers,
    y_val_no_outliers,
)
xgb_results


,learning_rate,max_depth,n_estimators,val_rmse
0,0.10,6,600,576634.411894
1,0.10,6,300,578808.679852
2,0.15,6,300,579791.223081
3,0.05,6,600,579876.978253
4,0.15,6,600,582739.090384
5,0.15,7,600,585625.691365
6,0.15,7,300,586173.247462
7,0.15,4,600,586847.874875
8,0.05,6,300,588416.213809
9,0.10,5,600,589578.490183


In [31]:
comparison_results = {}

comparison_results["Baseline (with outliers)"] = evaluate(
    y_val, xgb_base.predict(X_val), "XGB base w/ outliers"
)

comparison_results["Baseline (no outliers)"] = evaluate(
    y_val_no_outliers, xgb_base_no_outliers.predict(X_val_no_outliers), "XGB base clean"
)

best_params = xgb_results.iloc[0][
    ["max_depth", "learning_rate", "n_estimators"]
].to_dict()
best_params["max_depth"] = int(best_params["max_depth"])
best_params["n_estimators"] = int(best_params["n_estimators"])

xgb_tuned = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params)
xgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)
comparison_results["Tuned (no outliers)"] = evaluate(
    y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB tuned"
)


XGB base w/ outliers | RMSE: $8,237,289  MAE: $284,916  R2: 0.0236  MAPE: 16.11%
XGB base clean | RMSE: $578,809  MAE: $181,277  R2: 0.8373  MAPE: 11.95%
 XGB tuned | RMSE: $576,634  MAE: $176,148  R2: 0.8385  MAPE: 11.57%


In [32]:
comparison_df = pd.DataFrame(comparison_results).T
comparison_df["rmse"] = comparison_df["rmse"].map(lambda x: f"${x:,.0f}")
comparison_df["mae"] = comparison_df["mae"].map(lambda x: f"${x:,.0f}")
comparison_df["r2"] = comparison_df["r2"].map(lambda x: f"{x:.4f}")
comparison_df["mape"] = comparison_df["mape"].map(lambda x: f"{x:.2f}%")
comparison_df


,rmse,mae,r2,mape
Baseline (with outliers),"$8,237,289","$284,916",0.0236,16.11%
Baseline (no outliers),"$578,809","$181,277",0.8373,11.95%
Tuned (no outliers),"$576,634","$176,148",0.8385,11.57%


## LightGBM

In [33]:
lgb_base = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_base.fit(X_train, y_train)
evaluate(y_val, lgb_base.predict(X_val), "LGBM base")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003143 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3477
[LightGBM] [Info] Number of data points in the train set: 129477, number of used features: 35
[LightGBM] [Info] Start training from score 13.777438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'rmse': 8237850.519271379,
 'mae': 287350.32955997874,
 'r2': 0.023478738253345566,
 'mape': np.float64(16.248495671501605)}

In [34]:
lgb_base_no_outliers = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lgb_base_no_outliers.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    lgb_base_no_outliers.predict(X_val_no_outliers),
    "LGBM base without outliers",
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003215 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

{'rmse': 578443.8767571585,
 'mae': 184798.65965209858,
 'r2': 0.837483606669243,
 'mape': np.float64(12.209129602053066)}

In [35]:
def grid_search_lgb(param_grid, X_train, y_train, X_val, y_val):
    results = []
    for params in ParameterGrid(param_grid):
        model = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rmse = root_mean_squared_error(np.exp(y_val), np.exp(pred))
        results.append({**params, "val_rmse": rmse})
    return pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)


param_grid = {
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "n_estimators": [300, 600],
}

lgb_results = grid_search_lgb(
    param_grid,
    X_train_no_outliers,
    y_train_no_outliers,
    X_val_no_outliers,
    y_val_no_outliers,
)
lgb_results


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003302 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

,learning_rate,max_depth,n_estimators,val_rmse
0,0.15,7,600,563212.113016
1,0.15,7,300,565932.300437
2,0.15,6,300,568841.094821
3,0.10,6,600,569679.536084
4,0.15,5,600,572817.941663
5,0.15,5,300,572980.434061
6,0.15,4,600,574216.284533
7,0.15,6,600,574783.071269
8,0.05,7,600,577964.159461
9,0.10,7,600,578232.956126


In [36]:
comparison_results_lgb = {}

comparison_results_lgb["Baseline (with outliers)"] = evaluate(
    y_val, lgb_base.predict(X_val), "LGBM base w/ outliers"
)

comparison_results_lgb["Baseline (no outliers)"] = evaluate(
    y_val_no_outliers,
    lgb_base_no_outliers.predict(X_val_no_outliers),
    "LGBM base clean",
)

best_params_lgb = lgb_results.iloc[0][
    ["max_depth", "learning_rate", "n_estimators"]
].to_dict()
best_params_lgb["max_depth"] = int(best_params_lgb["max_depth"])
best_params_lgb["n_estimators"] = int(best_params_lgb["n_estimators"])

lgb_tuned = lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params_lgb)
lgb_tuned.fit(X_train_no_outliers, y_train_no_outliers)
comparison_results_lgb["Tuned (no outliers)"] = evaluate(
    y_val_no_outliers, lgb_tuned.predict(X_val_no_outliers), "LGBM tuned"
)


LGBM base w/ outliers | RMSE: $8,237,851  MAE: $287,350  R2: 0.0235  MAPE: 16.25%
LGBM base clean | RMSE: $578,444  MAE: $184,799  R2: 0.8375  MAPE: 12.21%
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003556 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3475
[LightGBM] [Info] Number of data points in the train set: 128764, number of used features: 35
[LightGBM] [Info] Start training from score 13.779382
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [37]:
comparison_df_lgb = pd.DataFrame(comparison_results_lgb).T
comparison_df_lgb["rmse"] = comparison_df_lgb["rmse"].map(lambda x: f"${x:,.0f}")
comparison_df_lgb["mae"] = comparison_df_lgb["mae"].map(lambda x: f"${x:,.0f}")
comparison_df_lgb["r2"] = comparison_df_lgb["r2"].map(lambda x: f"{x:.4f}")
comparison_df_lgb["mape"] = comparison_df_lgb["mape"].map(lambda x: f"{x:.2f}%")
comparison_df_lgb


,rmse,mae,r2,mape
Baseline (with outliers),"$8,237,851","$287,350",0.0235,16.25%
Baseline (no outliers),"$578,444","$184,799",0.8375,12.21%
Tuned (no outliers),"$563,212","$176,711",0.8459,11.65%


In [38]:
final_comparison = {
    "XGBoost (tuned)": evaluate(
        y_val_no_outliers, xgb_tuned.predict(X_val_no_outliers), "XGB tuned"
    ),
    "LightGBM (tuned)": evaluate(
        y_val_no_outliers, lgb_tuned.predict(X_val_no_outliers), "LGBM tuned"
    ),
}

final_comparison_df = pd.DataFrame(final_comparison).T
final_comparison_df["rmse"] = final_comparison_df["rmse"].map(lambda x: f"${x:,.0f}")
final_comparison_df["mae"] = final_comparison_df["mae"].map(lambda x: f"${x:,.0f}")
final_comparison_df["r2"] = final_comparison_df["r2"].map(lambda x: f"{x:.4f}")
final_comparison_df["mape"] = final_comparison_df["mape"].map(lambda x: f"{x:.2f}%")
final_comparison_df


 XGB tuned | RMSE: $576,634  MAE: $176,148  R2: 0.8385  MAPE: 11.57%
LGBM tuned | RMSE: $563,212  MAE: $176,711  R2: 0.8459  MAPE: 11.65%


,rmse,mae,r2,mape
XGBoost (tuned),"$576,634","$176,148",0.8385,11.57%
LightGBM (tuned),"$563,212","$176,711",0.8459,11.65%


In [39]:
RANDOM_STATE = 20260805


def evaluate(y_true_log, y_pred_log, label=""):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(
        f"{label:>10} | RMSE: ${rmse:,.0f}  MAE: ${mae:,.0f}  R2: {r2:.4f}  MAPE: {mape:.2f}%"
    )
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}


from sklearn.tree import DecisionTreeRegressor

dt_check = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)
dt_check.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(y_val_no_outliers, dt_check.predict(X_val_no_outliers), "DT check")


  DT check | RMSE: $699,354  MAE: $236,600  R2: 0.7624  MAPE: 15.78%


{'rmse': 699354.3651372286,
 'mae': 236600.1564947272,
 'r2': 0.762442197475818,
 'mape': np.float64(15.779801717203569)}

In [40]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer


def r2_on_price_scale(y_true_log, y_pred_log):
    return r2_score(np.exp(y_true_log), np.exp(y_pred_log))


price_scale_r2 = make_scorer(r2_on_price_scale, greater_is_better=True)
tscv = TimeSeriesSplit(n_splits=5)

param_grid_dt = {"max_depth": [3, 5, 7, 10, 15, 20, None]}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid_dt,
    cv=tscv,
    scoring=price_scale_r2,
    n_jobs=-1,
)
grid_search_dt.fit(X_train_no_outliers, y_train_no_outliers)
print(f"Best max_depth: {grid_search_dt.best_params_}")

dt_tuned = DecisionTreeRegressor(
    max_depth=grid_search_dt.best_params_["max_depth"],
    random_state=RANDOM_STATE,
)
dt_tuned.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    dt_tuned.predict(X_val_no_outliers),
    "DT tuned",
)


Best max_depth: {'max_depth': 7}
  DT tuned | RMSE: $733,079  MAE: $261,653  R2: 0.7390  MAPE: 17.64%


{'rmse': 733079.2234448927,
 'mae': 261652.85078180063,
 'r2': 0.738978343395463,
 'mape': np.float64(17.64157681564917)}

In [41]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15, 20, None],
    "max_features": ["sqrt", "log2", 0.5],
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=tscv,
    scoring=price_scale_r2,
    n_jobs=1,
)
grid_search_rf.fit(X_train_no_outliers, y_train_no_outliers)
print(f"Best params: {grid_search_rf.best_params_}")

rf_tuned = RandomForestRegressor(
    **grid_search_rf.best_params_, random_state=RANDOM_STATE, n_jobs=-1
)
rf_tuned.fit(X_train_no_outliers, y_train_no_outliers)
evaluate(
    y_val_no_outliers,
    rf_tuned.predict(X_val_no_outliers),
    "RF tuned",
)


Best params: {'max_depth': None, 'max_features': 0.5, 'n_estimators': 300}
  RF tuned | RMSE: $597,341  MAE: $183,126  R2: 0.8267  MAPE: 11.94%


{'rmse': 597341.182163079,
 'mae': 183126.127804955,
 'r2': 0.8266915917148114,
 'mape': np.float64(11.943232260600738)}

In [42]:
final_train_val = {
    "Decision Tree (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(dt_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(dt_tuned.predict(X_val_no_outliers))
        ),
    },
    "Random Forest (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(rf_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(rf_tuned.predict(X_val_no_outliers))
        ),
    },
    "XGBoost (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(xgb_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(xgb_tuned.predict(X_val_no_outliers))
        ),
    },
    "LightGBM (tuned)": {
        "train_r2": r2_score(
            np.exp(y_train_no_outliers), np.exp(lgb_tuned.predict(X_train_no_outliers))
        ),
        "val_r2": r2_score(
            np.exp(y_val_no_outliers), np.exp(lgb_tuned.predict(X_val_no_outliers))
        ),
    },
}

train_val_df = pd.DataFrame(final_train_val).T
train_val_df["gap"] = train_val_df["train_r2"] - train_val_df["val_r2"]
train_val_df = train_val_df.round(4)
train_val_df


,train_r2,val_r2,gap
Decision Tree (tuned),0.7442,0.7390,0.0053
Random Forest (tuned),0.9558,0.8267,0.1291
XGBoost (tuned),0.9547,0.8385,0.1162
LightGBM (tuned),0.9390,0.8459,0.0931
